# SMA — render TTS qua Google Drive (Colab GPU)

Đường Drive cho `tools/colab_render.py`, dùng khi máy đang làm việc **không có `colabjob`** (ví dụ phiên Claude Code trên cloud).

1. Trên máy làm việc: `python3 tools/colab_render.py episodes/<series>/<ID> --bundle <thư mục>` tạo 3 file `sma_job.json`, `sma_runner.tar`, `remote_render.py`.
2. Đưa 3 file đó vào `MyDrive/Science_Media_Academy/colab_jobs/<JOB>/` (Claude có thể tự tải lên qua Google Drive connector).
3. Ở đây: **Runtime → Change runtime type → T4 GPU**, sửa `JOB` bên dưới, chạy hết các ô.
4. Notebook ghi `sma_out.tar` (FLAC, không mất chất lượng) vào cùng thư mục Drive. Trên máy làm việc: `python3 tools/colab_render.py episodes/<series>/<ID> --ingest <đường dẫn>/sma_out.tar`.

Notebook chỉ chạy đúng script `remote_render.py` mà pipeline sinh ra (cùng giọng, seed, watermark, phoneme override như đường `colabjob`). Nó từ chối chạy trên CPU. Không có token hay dữ liệu riêng nào trong job: chỉ lời đọc của kịch bản và 4 file mã runner.


In [ ]:
# Ô 1: kết nối Drive và kiểm tra job
JOB = "P001-test"   # tên thư mục con trong MyDrive/Science_Media_Academy/colab_jobs/

import os, shutil
from google.colab import drive
drive.mount('/content/drive')
JOB_DIR = f'/content/drive/MyDrive/Science_Media_Academy/colab_jobs/{JOB}'
need = ['sma_job.json', 'sma_runner.tar', 'remote_render.py']
missing = [f for f in need if not os.path.exists(os.path.join(JOB_DIR, f))]
if missing:
    raise SystemExit(f'Thiếu {missing} trong {JOB_DIR}')
import json
items = json.load(open(os.path.join(JOB_DIR, 'sma_job.json')))['items']
print(f'OK: job {JOB} có {len(items)} đoạn cần render')

In [ ]:
# Ô 2: render trên GPU (cùng script với đường colabjob)
for f in need:
    shutil.copy(os.path.join(JOB_DIR, f), '/content/' + f)
if os.path.exists('/content/sma_out'):
    shutil.rmtree('/content/sma_out')
if os.path.exists('/content/sma_out.tar'):   # never hand back a previous run's result
    os.remove('/content/sma_out.tar')
!python /content/remote_render.py

In [ ]:
# Ô 3: trả kết quả về Drive
out = '/content/sma_out.tar'
if not os.path.exists(out):
    raise SystemExit('Không có sma_out.tar: xem log ở ô 2')
shutil.copy(out, os.path.join(JOB_DIR, 'sma_out.tar'))
print('Đã ghi', os.path.join(JOB_DIR, 'sma_out.tar'), round(os.path.getsize(out) / 1e6, 1), 'MB')
print('Xong. Có thể ngắt runtime: Runtime → Disconnect and delete runtime (tiết kiệm compute unit).')